# Table of content

- **Introduction**
- **Objective**
- **Project Overview**
- **Reinforcement Learning (RL) and Q-Learning Foundations**
    * Q-Learning Advanced Topics
- **Reinforcement Learning (RL) and Q-Learning Implementation**
    * Setup Environment
    * Defining the Q-Learning Model
    * The Q-Learning Algorithm - The Operational Loop
    * Performance Evaluation
- **Advanced Deep Q-Learning Experiment**
    * Deep Q-Learning Experiment: The Impact of Network Architecture
    * Follow-up Experiment: Adaptive Exploration Rate ($\epsilon$-Greedy Scheduling)
    * Follow-up Experiment: Custom Reward Function (Reward Shaping)
- **Conclusion**

# Introduction

This academic tutorial notebook documents the step-by-step implementation and experimental analysis of the Deep Q-Learning (DQN) algorithm. As a foundational approach in Reinforcement Learning (RL), DQN successfully bridges the gap between neural networks and classic Q-Learning, enabling agents to tackle complex, high-dimensional environments.

# Objective

The central objective of this project is to train an agent to solve the CartPole-v1 environment—a classic control problem where the agent must balance a pole on a cart for the maximum possible duration. Beyond simple implementation, this notebook focuses on understanding how core design choices impact the learning process.

The entire DQN pipeline is systematically broken down, covering the following key areas:

- **Fundamental Setup:** Initialization of the environment, configuration of random seeds for reproducibility, and definition of the Neural Network Architecture (the Q-Network) to approximate the optimal action-value function, $Q ∗ (s,a)$.
- **Training Mechanism:** Implementation of crucial RL components like Experience Replay (using the memory buffer) and the ϵ-Greedy Strategy to manage the exploration-exploitation trade-off.
- **Experimental Analysis:** Targeted experiments are conducted to quantify the effects of:

  - **Network Capacity:** Testing the impact of a larger architecture (3×64).
  - **Adaptive Exploration:** Modifying ϵ decay based on agent performance.
  - **Reward Shaping:** Providing dense, continuous state feedback instead of sparse rewards.

This structured approach provides valuable insight into the practical challenges of DQN, offering clear results on stability, convergence, and the critical balance required for effective policy learning.

# Project Overview

This academic project implements and analyzes a complete Deep Q-Learning pipeline, focusing on training an agent to solve the classic CartPole-v1 control problem. The project's structure is modular, addressing environment configuration, model construction, hyperparameter tuning, and advanced experimental analysis.

**I. Methodology and Core Components**

The pipeline is built on the fundamental components of the DQN algorithm, ensuring stability and reproducibility:

- **Environment Setup:** The CartPole-v1 environment is initialized with 4 continuous state variables and 2 discrete actions. Crucial steps are taken to ensure reproducibility through the setting of all necessary random seeds (NumPy, TensorFlow, and Gym).
- **Neural Network Model:** A Deep Q-Network (DQN) is constructed using Keras/TensorFlow to approximate the optimal action-value function, $Q ∗ (s,a)$. The implementation utilizes two separate models - a main model for training and a target model for stable Q-value estimation in the Bellman equation - both trained using MSE loss and the Adam optimizer.
- **Training Loop & Stability:** Training relies on the Experience Replay mechanism, which stores transitions in a finite-size buffer (e.g., 2000). Action selection uses the ϵ-Greedy Policy, with ϵ decaying exponentially over time to balance exploration and exploitation.

**II. Targeted Experimental Analysis**
The project includes specific experiments designed to quantify the impact of key hyperparameter and design choices:

- **Network Capacity Study:** Performance is compared between a Baseline Architecture (2×32 neurons) and an Experimental Architecture (3×64 neurons) to analyze the effect of increased model capacity on learning efficiency and stability.
- **Adaptive Exploration:** An adaptive ϵ decay strategy is tested, where the exploration rate is reduced faster only after the agent achieves a high-performance threshold (e.g., score ≥200). This aims to investigate dynamic exploration scheduling.
- **Reward Shaping:** A custom, dense reward function is implemented. This function rewards the agent proportionally to its proximity to the optimal state (centered cart position, vertical pole angle), offering continuous feedback to accelerate learning compared to the default sparse reward.

**III. Robustness and Evaluation**

The pipeline incorporates robust error and warning handling to ensure compatibility across various Gym/Gymnasium API versions (handling both 4-tuple and 5-tuple step returns) and to mitigate common environment-related issues like rendering errors in headless environments.

The agent's performance is rigorously evaluated using a fully greedy policy (ϵ=0) over a fixed number of episodes, generating clear performance metrics for subsequent results analysis and discussion.

# Reinforcement Learning (RL) and Q-Learning Foundations

**Reinforcement Learning (RL)**

Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by interacting with an environment . The agent receives rewards or penalties for its actions and aims to maximize cumulative rewards over time.

- Environment: The world the agent interacts with (e.g., CartPole-v1).
- State (s): Current situation of the environment (e.g., cart position, pole angle).
- Action (a): Choices the agent can make (e.g., push cart left or right).
- Reward (r): Feedback signal after an action (positive for good behavior, negative for bad).
- Policy (π): Strategy for choosing actions based on states.
- Episode: A sequence of states, actions, and rewards until a terminal state (e.g., pole falls).

**Q-Learning Algorithm**

The **Q-Learning algorithm** is a **model-free** and **off-policy** reinforcement learning (RL) technique. It is designed to find an **optimal policy** ($\pi^*$)—a set of rules for what action to take in a given state—by learning an **action-value function**, typically called the **Q-function** or $Q(s, a)$.

**Core Concepts**

* **Model-Free:** This means the algorithm does not require prior knowledge of the environment's internal mechanics, such as the probabilities of transitioning from one state to another ($P(s' | s, a)$) or the precise reward structure. It learns purely through interaction with the environment.
* **Off-Policy:** The algorithm learns the optimal policy ($\pi^*$) and its associated Q-values by observing the outcomes of actions chosen by a *different* policy, often a more **exploratory** policy (like $\epsilon$-greedy). This separation of the learning policy and the action-selection policy is what makes it "off-policy."
* **Q-Function ($Q(s, a)$):** This function represents the **maximum expected future reward** (or return) an agent can achieve by starting in state $s$ and taking action $a$, and then following the optimal policy thereafter.

**The Update Rule (Bellman Equation)**

The heart of Q-Learning is its iterative update rule, which is derived from the **Bellman Equation**. Q-Learning is a model-free RL algorithm that learns an action-value function Q(s, a), which estimates the expected future rewards for taking action a in state s. It uses the Bellman equation:

$ Q(s, a) = r + \gamma \max_{a'} Q(s', a') $

Where:

- $ r $ is the immediate reward.
- $ \gamma $ (discount factor) is typically 0.95, prioritizing immediate over future rewards.
- $ s' $ is the next state.
- $ \max_{a'} Q(s', a') $ is the best Q-value in the next state.

Therefore, the agent uses this equation to update its estimate of the $Q(s, a)$ value based on the experience of an interaction ($s, a, r, s'$):

$$ Q(s, a) \leftarrow (1 - \alpha) Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') \right] $$

Where:

* $Q(s, a)$: The current estimate of the Q-value for the current state $s$ and action $a$.
* $\alpha$ (Learning Rate): A value between 0 and 1 that determines how much the new information will override the old information. $\alpha=0$ means the Q-values are never updated; $\alpha=1$ means the agent only considers the most recent experience.
* $r$ (Reward): The immediate reward received after taking action $a$ in state $s$ and transitioning to the next state $s'$.
* $\gamma$ (Discount Factor): A value between 0 and 1 that determines the importance of future rewards. A $\gamma$ close to 0 makes the agent "myopic" (only caring about immediate rewards), while a $\gamma$ close to 1 makes the agent strive for long-term high rewards.
* $\max_{a'} Q(s', a')$: The maximum expected future reward from the **next state $s'$**, assuming the agent chooses the best possible action $a'$ from that point forward. This represents the optimal value of the next state.
* $\left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$ (Temporal Difference Error, or TD Error): The difference between the newly observed value (the "target," $r + \gamma \max_{a'} Q(s', a')$) and the current estimate ($Q(s, a)$). The algorithm uses this error to adjust its current Q-value estimate.

In deep Q-Learning (DQN), a neural network approximates Q-values, handling high-dimensional states better than tabular methods.

**Operation Steps**

The basic Q-Learning process involves repeatedly executing the following steps:

1.  **Initialize:** Create and initialize a Q-table (a matrix where rows are states and columns are actions) with arbitrary values (often zero).
2.  **Observe State:** The agent observes the current state $s$.
3.  **Select Action:** The agent selects an action $a$ based on a policy derived from the current Q-table (e.g., using an $\epsilon$-greedy strategy, where it mostly chooses the action with the highest Q-value but occasionally chooses a random action for exploration).
4.  **Execute Action:** The agent executes action $a$, receives an immediate reward $r$, and transitions to the next state $s'$.
5.  **Update Q-Value:** The agent updates the $Q(s, a)$ value using the Bellman equation shown above.
6.  **Loop:** Set $s \leftarrow s'$ and repeat the process until the task is complete or a set number of episodes is finished.

**Optimal Policy Derivation**

Once the Q-table has converged (the Q-values are no longer changing significantly), the **optimal policy** ($\pi^*$) can be easily derived. For any given state $s$, the optimal action $a^*$ is simply the action that maximizes the Q-value:

$$
\pi^*(s) = \arg\max_a Q(s, a)
$$

This means the agent will choose the action that leads to the highest estimated long-term reward.

## Q-Learning Advanced Topics

**Exploration vs. Exploitation ($\epsilon$-Greedy Policy)**

Effective RL requires balancing two conflicting goals:

- Exploitation: Taking the action $a$ that currently has the highest estimated Q-value, $a=\arg\max_a Q(s,a)$. This maximizes immediate reward based on current knowledge.
- Exploration: Taking a random action $a$ to discover new, potentially better paths and rewards in the environment.

Q-Learning manages this through the $\epsilon$-Greedy Policy:

- With a small probability $\epsilon$ (epsilon), the agent chooses a random action (Explore).
- With probability $1-\epsilon$, the agent chooses the action with the maximum Q-value (Exploit).

In this implementation, $\epsilon$ starts high (e.g., $1.0$) and decays over time (e.g., multiplied by $0.99$ after each step). This dynamic approach favors extensive exploration at the start, when Q-values are unreliable, and transitions to exploitation as the agent's knowledge stabilizes.

**Deep Q-Networks (DQN)**

Traditional Q-Learning uses a Q-table to store $Q(s,a)$ values for every possible state-action pair. This is infeasible for environments like CartPole, which have continuous state spaces (cart position, velocity, etc.), resulting in an infinite number of states. Deep Q-Learning (DQN) addresses this by replacing the Q-table with a Neural Network (NN), specifically a Deep Q-Network.

- **Input:** The current state $s$.
- **Output:** The Q-value for every possible action $a$.
- **Function Approximation:** The NN acts as a function approximator, learning a function $Q(s,a;\theta)\approx Q^*(s,a)$, where $\theta$ represents the network's weights. This allows the model to generalize Q-values to unseen states.

The training process for the DQN involves several stabilization techniques:

- **Experience Replay:** This technique involves storing the agent's experiences (a tuple of $(s,a,r,s',\text{done})$) in a fixed-size memory buffer (e.g., $2000$ transitions). This addresses two major issues:
- **Breaking Temporal Correlations:** Experiences occur in sequence, leading to highly correlated training data. Randomly sampling batches from the buffer breaks these correlations, improving training stability.
- **Increased Data Efficiency:** Each stored experience can be reused for multiple updates. The buffer has a fixed size (e.g., $2000$), which ensures that older, less relevant experiences are naturally forgotten, focusing learning on recent interactions.
- **Batch Training:** Training is conducted over episodes, where the network update is performed after specific steps. This involves sampling batches of transitions from the replay buffer. The use of batched data leverages the efficiency of vectorized operations available in numerical libraries like NumPy and machine learning frameworks like TensorFlow/Keras.

The NN is optimized by minimizing the squared error between the predicted Q-value and the target Q-value: the immediate reward plus the discounted maximum future Q-value, as dictated by the Bellman Equation. The network weights are adjusted using backpropagation to reduce this loss, iteratively refining the Q-value estimates.

# Reinforcement Learning (RL) and Q-Learning Implementation

## Setup Environment

Before implementing the algorithm, the simulation environment must be established. OpenAI Gym provides standardized environments for RL testing.

**CartPole-v1 Environment**

CartPole-v1 is a classic control problem :

- **State Space:** Defined by 4 continuous values (cart position, cart velocity, pole angle, pole angular velocity).
- **Action Space:** Consists of 2 discrete actions (push cart left or push cart right).
- **Goal:** To keep the pole balanced for as long as possible (maximum 200 or 500 steps per episode, depending on the environment version).
- **Terminal States:** The episode ends if the pole angle exceeds $\pm 12^\circ$, the cart position moves beyond $±2.4$, or the episode length is exceeded.

Setting random seeds ensures reproducibility. This is crucial as RL is stochastic due to random exploration and inherent environment dynamics. Addressing potential library issues, such as TensorFlow warnings or recursion limits, is also necessary to ensure smooth execution.

**Library Installation and Configuration**

The necessary libraries for the reinforcement learning setup must be installed, and configuration adjustments may be applied for smooth execution, especially concerning dependencies like TensorFlow. The following commands install the required environment library and ensure a specific version of NumPy is used for compatibility.

**Packages Installation**

In [ ]:
!pip install gym

In [ ]:
# Suppress warnings for a cleaner notebook or console experience
import warnings
warnings.filterwarnings('ignore')

# Suppress warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Override the default warning function
def warn(*args, **kwargs):
    pass
warnings.warn = warn

# Import necessary libraries for the Q-Learning model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input  # Import Input layer
from tensorflow.keras.optimizers import Adam
import gym  
from collections import deque
import random
import numpy as np
import tensorflow as tf

**Environment Variable Adjustments**

Environment variables can be set to mitigate potential conflicts or optimize performance with machine learning frameworks. Disabling oneDNN optimizations or restricting GPU visibility can simplify local execution.

In [2]:
# Set environment variables to mitigate TensorFlow issues
import os 
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

**Recursion Limit**

The system recursion limit is adjusted. Although this is typically a workaround, it can prevent deep recursive calls encountered in some Python implementations from raising errors.

In [3]:
import sys 
sys.setrecursionlimit(1500)

**Environment Initialization**

The RL environment is initialized using the imported libraries.

In [5]:
import gym  # Ensure the environment library is available
# Create the environment 
env = gym.make('CartPole-v1') 

- **gym:** A foundational toolkit for developing and comparing reinforcement learning algorithms.
- **CartPole-v1:** A classic control problem . The objective is to apply horizontal forces to a cart to prevent a pole attached to it from falling over.

**Ensuring Reproducibility**

Random seeds are set for the numerical library, the environment's action space, and the observation space. This practice is essential in stochastic environments to guarantee that the results obtained are reproducible across different runs.

In [6]:
# Set random seeds for reproducibility 
np.random.seed(42) 
env.action_space.seed(42) 
env.observation_space.seed(42)

[42]

## Defining the Q-Learning Model

In Q-Learning, a neural network is used as a function approximator for $Q(s,a)$, which is referred to as a Deep Q-Network (DQN).

**Neural Network Architecture**

A simple feedforward network is typically used, featuring:

- **Input Layer:** Matches the dimensionality of the state space (4 inputs for CartPole).
- **Hidden Layers:** Consist of dense layers utilizing the Rectified Linear Unit (ReLU) activation function. ReLU introduces the necessary non-linearity, enabling the network to learn complex mappings between continuous state inputs and discrete Q-value outputs.
- **Output Layer:** A dense layer with linear activation, producing a Q-value for every possible action (2 outputs for the CartPole environment). The linear activation allows the output values to represent raw, unbounded expected rewards.
- **Loss Function:** The Mean Squared Error (MSE) is employed to minimize the difference between the network's predicted Q-values and the Target Q-values (calculated via the Bellman Equation). Minimizing this error forces the predicted Q-values to converge toward the optimal $Q^∗$ values.
- **Optimizer:** Adam is an adaptive learning rate optimization algorithm used for efficient training.
- **Keras:** This high-level API is utilized for the quick and modular prototyping of the neural network architecture.

The resulting model predicts Q-values for all available actions given the current state $s$. This prediction capability is central to the agent's decision-making process, allowing it to select the action with the highest Q-value (exploitation) or a random one (exploration).

In [7]:
# Define state and action sizes
state_size = env.observation_space.shape[0] # State is the observation space size
action_size = env.action_space.n # Action is the number of possible actions (2 for CartPole)

# Define the model building function
def build_model(state_size, action_size): 
    """Creates the Deep Q-Network (DQN) model."""
    model = Sequential() 
    # Input layer defined by the state dimensionality
    model.add(Input(shape=(state_size,))) 
    # Hidden layers use ReLU for non-linearity
    model.add(Dense(24, activation='relu')) 
    model.add(Dense(24, activation='relu')) 
    # Output layer provides Q-values for each action using linear activation
    model.add(Dense(action_size, activation='linear')) 
    # Compile with MSE loss and Adam optimizer (minimizing error between predicted and target Q-values)
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001)) 
    return model 

# Build main and target models (for Double DQN)
model = build_model(state_size, action_size)
target_model = build_model(state_size, action_size)
target_model.set_weights(model.get_weights())  # Initialize target network with same weights

2025-10-01 20:48:29.251704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759351709.278753    2323 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759351709.287204    2323 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-01 20:48:33.460168: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## The Q-Learning Algorithm - The Operational Loop

The execution of the Deep Q-Network (DQN) algorithm is governed by three critical, interconnected mechanisms that ensure efficient and stable learning:

**1. Experience Replay**

* **Mechanism:** Experiences—defined as the transition tuple $(\mathbf{s}, \mathbf{a}, \mathbf{r}, \mathbf{s'}, \mathbf{done})$—are stored in a **Replay Buffer**, typically implemented as a fixed-size queue (e.g., 2000 transitions).
* **Theoretical Impact:** Storage breaks the **temporal correlations** inherent in sequential exploration. Randomly sampling from this buffer for training batches significantly improves the stability of the neural network's gradient updates, mitigating the risk of catastrophic forgetting. The fixed size ensures that older, potentially less relevant experiences are systematically discarded.

**2. Epsilon-Greedy Action Selection**

* **Trade-off:** The agent must balance **exploration** (discovering better paths) and **exploitation** (leveraging current best knowledge).
* **Mechanism:** An **$\epsilon$-Greedy Policy** is used. With a probability of $\epsilon$, a random action is taken (exploration). With a probability of $1 - \epsilon$, the action with the maximum estimated Q-value from the neural network is chosen (exploitation).
* **Decay:** The value of $\epsilon$ starts high (e.g., $1.0$) to promote initial exploration and is systematically decayed (e.g., multiplied by $0.99$ after each step). This decay schedule shifts the agent's focus from exploration to exploitation as knowledge matures.

**3. Network Training and Update**

* **Target Calculation:** Training involves sampling a **batch** of stored experiences from the Replay Buffer. The **Target Q-value** for each experience is computed using the **Bellman Equation** and the **Discount Factor ($\gamma$)**.
    * The **Discount Factor** ($\gamma=0.95$ in this case) weights the expected value of future rewards, determining the agent's time horizon.
* **Efficiency:** **Batch Training** is highly efficient as it leverages the vectorized operations available in frameworks like NumPy and TensorFlow.
* **Learning:** The difference between the network's current predicted Q-value and the computed Target Q-value defines the error. This error is minimized using the **Mean Squared Error (MSE) loss function**, and the network weights are updated via **backpropagation** and the **Adam optimizer**. This process iteratively nudges the network towards approximating the optimal Q-function, $Q^*$.

In [ ]:
# Define hyperparameters

# Initial exploration rate: Agent starts by choosing random actions
epsilon = 1.0 
# Minimum exploration rate: Ensures a small amount of exploration continues
epsilon_min = 0.01
# Rate at which epsilon decreases after each training step (controls exploration-exploitation trade-off)
epsilon_decay = 0.995
# Experience Replay Memory: Stores past experiences (state, action, reward, next_state, done)
memory = deque(maxlen=2000)
# Discount factor: Determines the importance of future rewards (closer to 1.0 means more value on long-term rewards)
gamma = 0.95 
# Frequency (in episodes) to copy weights from the main model to the target model (DQN innovation for stability)
target_update_frequency = 10

def remember(state, action, reward, next_state, done):
    """Store experience in memory.""" 
    # Stores the transition tuple (s, a, r, s', done) into the replay buffer
    memory.append((state, action, reward, next_state, done))

def replay(batch_size=64):
    """Train the model using a random sample of experiences.
       DQN training function using Experience Replay""" 
    if len(memory) < batch_size:
        # Wait until memory has enough experiences to form a batch
        return 
    # Sample a random batch of experiences
    minibatch = random.sample(memory, batch_size) 
    # Separate the components of the experience batch into NumPy arrays for model processing
    states = np.vstack([x[0] for x in minibatch])
    actions = np.array([x[1] for x in minibatch])
    rewards = np.array([x[2] for x in minibatch])
    next_states = np.vstack([x[3] for x in minibatch])
    dones = np.array([x[4] for x in minibatch])
    
    with tf.device('/CPU:0'):
        # Use target model (fixed/stable weights) to estimate max Q-value of next state (Q(s', a'))
        q_next = target_model.predict(next_states, verbose=0) 
        # Get current Q-values for the states from the main (trained) model
        q_target = model.predict(states, verbose=0) 
    
    for i in range(batch_size):
        # Start target with the immediate reward (r)
        target = rewards[i]
        
        if not dones[i]: 
            # Bellman Equation: r + gamma * max(Q(s', a'))
            # Calculate the discounted maximum expected future reward
            target += gamma * np.amax(q_next[i]) 
        # Update the target Q-value *only* for the action taken
        q_target[i][actions[i]] = target 

    # Train the main model on the states and the newly calculated target Q-values
    model.fit(states, q_target, epochs=1, verbose=0) 
    global epsilon
    if epsilon > epsilon_min:
        # Decay epsilon after training step
        epsilon *= epsilon_decay

def act(state):
    """Epsilon-greedy action selection.
       Agent decision-making policy"""
    if np.random.rand() <= epsilon:
        # Explore: choose a random action
        return random.randrange(action_size) 
    with tf.device('/CPU:0'):
        # Exploit: query the main model for Q-values
        act_values = model.predict(state, verbose=0)
    # Return the action with the highest predicted Q-value
    return np.argmax(act_values[0]) 

# Training loop

# Total number of training episodes to run
episodes = 30 
# Frequency (in steps) to perform a replay training operation
train_frequency = 5 

# Iterate through each episode
for e in range(episodes): 
    # Reset environment for a new episode
    state = env.reset() 
    # Reshape state for model input (batch size 1)
    state = np.reshape(state, [1, state_size]) 

    # Max steps per episode
    for time in range(200): 
        # Select action based on epsilon-greedy policy
        action = act(state) 
        # Execute action in the environment
        next_state, reward, done, _ = env.step(action) 
        # Apply penalty reward if the episode terminates early
        reward = reward if not done else -10 
        # Reshape next state
        next_state = np.reshape(next_state, [1, state_size]) 
        # Store the experience
        remember(state, action, reward, next_state, done) 
        # Move to the next state
        state = next_state 

        # Check if episode finished
        if done: 
            # Print episode summary
            print(f"episode: {e+1}/{episodes}, score: {time}, e: {epsilon:.2f}") 
            break
        if time % train_frequency == 0:
            # Perform training replay periodically
            replay(batch_size=64) 
    
    # Update target model periodically
    if (e + 1) % target_update_frequency == 0:
        # Update the target network weights for stable targets
        target_model.set_weights(model.get_weights()) 

episode: 1/30, score: 42, e: 1.00
episode: 2/30, score: 17, e: 1.00
episode: 3/30, score: 21, e: 0.98
episode: 4/30, score: 27, e: 0.95
episode: 5/30, score: 9, e: 0.94
episode: 6/30, score: 16, e: 0.92
episode: 7/30, score: 10, e: 0.91
episode: 8/30, score: 29, e: 0.89
episode: 9/30, score: 10, e: 0.88
episode: 10/30, score: 22, e: 0.86
episode: 11/30, score: 14, e: 0.84
episode: 12/30, score: 13, e: 0.83
episode: 13/30, score: 15, e: 0.82
episode: 14/30, score: 38, e: 0.79
episode: 15/30, score: 44, e: 0.75
episode: 16/30, score: 16, e: 0.74
episode: 17/30, score: 15, e: 0.73
episode: 18/30, score: 11, e: 0.71
episode: 19/30, score: 12, e: 0.70
episode: 20/30, score: 8, e: 0.70
episode: 21/30, score: 11, e: 0.69
episode: 22/30, score: 11, e: 0.68
episode: 23/30, score: 10, e: 0.67
episode: 24/30, score: 10, e: 0.66
episode: 25/30, score: 8, e: 0.66
episode: 26/30, score: 10, e: 0.65
episode: 27/30, score: 14, e: 0.64
episode: 28/30, score: 10, e: 0.63
episode: 29/30, score: 9, e: 0.6

## Performance Evaluation

After the training phase concludes, the performance of the learned policy must be quantified. Evaluation shifts from the dynamic, exploratory environment of training to a deterministic, greedy mode.

**Pure Exploitation:** 

The core of evaluation involves running episodes where the agent acts purely greedily. The ϵ-greedy policy is deactivated, meaning the exploration rate ϵ is set to zero. The agent selects only the action a that yields the maximum predicted Q-value for the current state  $s$, i.e., $a=\arg\max_a Q(s,a)$.

**Performance Metrics and Generalization:** 

The primary performance metric in the CartPole environment is the number of steps the pole remains balanced (the reward accumulated per episode). A higher score indicates a more effective and stable policy. Evaluation serves as a test of the policy's generalization—its ability to apply the learned knowledge to new, unseen sequences of states. Visualization of the environment during evaluation (rendering) provides intuitive confirmation of the agent's learned strategy.

**Handling Stochasticity and Convergence:** 

To obtain a statistically reliable measure of the agent's skill, performance is typically averaged over a significant number of evaluation episodes (e.g., 100 episodes). This averaging mitigates the influence of any remaining stochasticity in the environment's initialization or dynamics, providing a robust measure of the policy's expected return. This average reward is the true indicator of whether the model has converged to an effective or "solved" policy.

In [11]:
# Evaluation loop
# Run 10 evaluation episodes
for e in range(10): 
    # Reset environment (new Gym version returns a tuple)
    state = env.reset() 
    # Reshape state
    state = np.reshape(state, [1, state_size])
    # Max evaluation steps
    for time in range(500):
        # Display the environment visually
        env.render()
        with tf.device('/CPU:0'):
            # Greedy action selection (Exploitation only, epsilon is ignored)
            action = np.argmax(model.predict(state, verbose=0)[0])
        # Take the step (old Gym return format)
        next_state, reward, done, _ = env.step(action) 
        next_state = np.reshape(next_state, [1, state_size])
        state = next_state
        if done:
            print(f"episode: {e+1}/10, score: {time}")
            break

# Close the environment window after evaluation
env.close()

error: XDG_RUNTIME_DIR not set in the environment.


episode: 1/10, score: 9
episode: 2/10, score: 8
episode: 3/10, score: 8
episode: 4/10, score: 8
episode: 5/10, score: 9
episode: 6/10, score: 8
episode: 7/10, score: 8
episode: 8/10, score: 8
episode: 9/10, score: 8
episode: 10/10, score: 8


- This loop runs 10 episodes to test the trained agent.
- env.render(): visualizes the environment.
- The agent chooses actions based on the trained model and interacts with the environment.
- The score for each episode is printed.

# Advanced Deep Q-Learning Experiment

**Advanced Deep Q-Learning Experiment: Architecture, Exploration, and Reward Shaping**

With this tutorial notebook, it details three key experiments designed to analyze the impact of design choices—network architecture, exploration scheduling, and reward function design—on the performance of a Deep Q-Learning (DQL) agent operating in the CartPole-v1 environment.

## Deep Q-Learning Experiment: The Impact of Network Architecture

**Theoretical Background: Function Approximation and Capacity**

The core of Deep Q-Learning is the use of a Neural Network (Q-Network) to approximate the optimal action-value function, $Q^*$(s,a). This network acts as a **non-linear function approximator**.

**Model Capacity and the Bias-Variance Trade-off**

The architecture of the Q-Network (number of layers and neurons) determines its **model capacity**.

- **Low Capacity (High Bias):** A smaller, shallower network has limited ability to represent complex relationships in the state space. This results in **high bias**, meaning the model consistently fails to capture the true underlying $Q^*$ function, leading to **underfitting** and suboptimal policy performance.
- **High Capacity (High Variance):** A larger, deeper network can potentially learn highly complex functions (**low bias**). However, if the training data (stored in the replay buffer) is limited or noisy, the model may fit the random noise in the samples too closely. This results in **high variance**, where minor changes in the input data lead to large changes in the output prediction, causing **overfitting** and instability.

The goal in network design is to find the minimum capacity necessary to effectively capture the environmental dynamics without introducing excessive variance.

**Role of Components**

- **Dense Layers \& ReLU:** The use of **Dense layers** performs linear transformations, while the **Rectified Linear Unit (ReLU)** activation function introduces the crucial **non-linearity** necessary for the network to approximate complex, non-linear functions, as required by the Bellman equation.
- **Adam Optimizer \& MSE Loss:** The model is trained using **Mean Squared Error (MSE) loss**, minimizing the difference between the predicted Q-value and the target Q-value (derived from the Bellman equation). The **Adam optimizer** efficiently manages adaptive learning rates for each network weight.

This content is now ready to be pasted directly into a markdown cell in your Jupyter Notebook.

**Experiment Objective and Hypothesis**

- **Objective:** To quantify the change in the Q-Learning agent's performance (measured by the average score over 100 episodes) when the capacity of the underlying Q-Network is substantially increased.
- **Hypothesis:** Increasing the network complexity from a baseline of two 32-neuron layers to an experimental architecture of three 64-neuron layers will lead to a faster convergence rate and a higher final average score in the CartPole-v1 environment, given that the environment's state-action complexity is likely manageable by the larger model without excessive overfitting.

**Baseline Architecture (Control Group)**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-2b7s{text-align:right;vertical-align:bottom}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Parameter</th>
    <th class="tg-7zrl">Value</th>
    <th class="tg-7zrl">Description</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Hidden Layers</td>
    <td class="tg-2b7s">2</td>
    <td class="tg-0lax">Number of hidden Dense layers.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Neurons/Layer</td>
    <td class="tg-2b7s">32</td>
    <td class="tg-0lax">Number of neurons in each hidden layer.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Total Hidden Neurons</td>
    <td class="tg-2b7s">64</td>
    <td class="tg-0lax">Calculated as 32×2.</td>
  </tr>
</tbody>
</table>

**Experimental Architecture (Test Group)**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-2b7s{text-align:right;vertical-align:bottom}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Parameter</th>
    <th class="tg-7zrl">Value</th>
    <th class="tg-7zrl">Description</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Hidden Layers</td>
    <td class="tg-2b7s">3</td>
    <td class="tg-0lax">Increased number of hidden Dense layers.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Neurons/Layer</td>
    <td class="tg-2b7s">64</td>
    <td class="tg-0lax">Increased number of neurons in each hidden layer.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Total Hidden Neurons</td>
    <td class="tg-2b7s">192</td>
    <td class="tg-0lax">Calculated as 64×3.</td>
  </tr>
</tbody>
</table>

**Network Architectures**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Group</th>
    <th class="tg-7zrl">Parameter</th>
    <th class="tg-7zrl">Value</th>
    <th class="tg-0lax">Theoretical Impact</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Baseline (Control)</td>
    <td class="tg-7zrl">Layers × Neurons</td>
    <td class="tg-7zrl">2×32</td>
    <td class="tg-0lax">Lower capacity, making it simpler and faster, but with a higher risk of bias (underfitting).</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Experimental (Test)</td>
    <td class="tg-7zrl">Layers × Neurons</td>
    <td class="tg-7zrl">3×64</td>
    <td class="tg-0lax">Higher capacity, enabling it to learn more complex functions. This potentially leads to lower bias but an increased variance risk (overfitting).</td>
  </tr>
</tbody>
</table>

**Performance Metric**

Performance will be measured by the mean of the scores (time steps survived) achieved in the last 100 training episodes.

In [ ]:
# Environment Setup
env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]  # 4 features
action_size = env.action_space.n          # 2 actions

# Global Settings
# NOTE: The instruction requires evaluating over 100 episodes.
episodes = 100 
batch_size = 32  
memory = deque(maxlen=2000)
gamma = 0.95  # Discount factor (hyperparameter for Q-Learning update)
epsilon = 1.0  # Exploration rate (hyperparameter for act function)
epsilon_min = 0.01
epsilon_decay = 0.995

# Core Model Definition (Implementation of Experimental Architecture)

def build_model(state_size, action_size):
    """
    Defines the Neural Network architecture for the Q-Network.
    This implementation uses the EXPERIMENTAL architecture: 3 layers x 64 neurons.
    
    To implement the BASELINE architecture, replace the 64-neuron layers with 32-neuron layers 
    and remove the third hidden layer.
    """
    model = Sequential()
    model.add(Input(shape=(state_size,)))    
    # EXPERIMENTAL ARCHITECTURE: Layer 1 (64 neurons)
    model.add(Dense(64, activation='relu')) 
    # EXPERIMENTAL ARCHITECTURE: Layer 2 (64 neurons)
    model.add(Dense(64, activation='relu'))
    # EXPERIMENTAL ARCHITECTURE: Layer 3 (64 neurons)
    model.add(Dense(64, activation='relu'))
    # Output layer provides Q-value for each action
    model.add(Dense(action_size, activation='linear'))
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

# Initialize the model with the experimental architecture
model = build_model(state_size, action_size)

# Agent Functions

# Epsilon-greedy action selection
def act(state):
    global epsilon
    if np.random.rand() <= epsilon:
        return env.action_space.sample()  # Explore (random action)
    # Exploit (predict best action from Q-Network)
    q_values = model.predict(state, verbose=0)
    return np.argmax(q_values[0])  

# Store the experience
def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

# Train the Q-Network using Experience Replay
def replay(batch_size):
    global epsilon
    if len(memory) < batch_size:
        return
        
    minibatch = random.sample(memory, batch_size)
    
    # Vectorize inputs for faster prediction
    states = np.vstack([sample[0] for sample in minibatch])
    next_states = np.vstack([sample[3] for sample in minibatch])
    
    # Predict Q-values for current and next states
    targets_f = model.predict(states, verbose=0)
    target_next = model.predict(next_states, verbose=0)
    
    # Calculate the target Q-value for the action taken (Bellman equation)
    for i, (state, action, reward, next_state, done) in enumerate(minibatch):
        target = reward if done else reward + gamma * np.amax(target_next[i])
        targets_f[i][action] = target
        
    # Perform one step of gradient descent
    model.fit(states, targets_f, epochs=1, verbose=0)
    
    # Decay epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

# Training Loop 

episode_scores = []
for e in range(1, episodes + 1):
    # Handle different Gym/Gymnasium reset() return signatures
    reset_result = env.reset()
    if isinstance(reset_result, tuple):
        state = reset_result[0]
    else:
        state = reset_result
    
    state = np.reshape(state, [1, state_size])
    time = 0
    done = False
    
    while not done and time < 500: # CartPole max steps is 500
        action = act(state)
        
        # Handle different Gym/Gymnasium step() return signatures ---
        step_result = env.step(action)
        
        if len(step_result) == 5:
            # New Gym/Gymnasium standard: (obs, reward, terminated, truncated, info)
            next_state, reward, terminated, truncated, _ = step_result
            done = terminated or truncated
        elif len(step_result) == 4:
            # Old Gym standard: (obs, reward, done, info)
            next_state, reward, done, _ = step_result
        else:
            # Fallback for unexpected number of returns (should not happen for CartPole)
            print(f"Warning: env.step() returned an unexpected number of values: {len(step_result)}. Assuming first element is state.")
            # Use tuple unpacking on a sliced result, ensuring we get at least the first 3 (state, reward, done)
            next_state, reward, done, *_ = step_result
        
        # Reward shaping: standard +1 reward, with penalty for failure
        reward = reward if not done else -10 
        
        next_state = np.reshape(next_state, [1, state_size])
        
        remember(state, action, reward, next_state, done)
        state = next_state
        time += 1
        
        # Train every 10 steps
        if len(memory) > batch_size and time % 10 == 0:  
            replay(batch_size) 
            
    episode_scores.append(time)
    
    # Print progress every 10 episodes
    if e % 10 == 0:
        avg_score = np.mean(episode_scores[-10:])
        print(f"Episode: {e}/{episodes}, Score: {time}, Avg(10): {avg_score:.2f}, Epsilon: {epsilon:.2f}")

final_average_score = np.mean(episode_scores)
print(f"\n--- EXPERIMENTAL ARCHITECTURE RESULTS (3x64) ---")
print(f"Total Episodes Run: {episodes}")
print(f"Overall Average Score: {final_average_score:.2f}")

env.close()

Episode: 10/100, Score: 49, Avg(10): 21.30, Epsilon: 0.93
Episode: 20/100, Score: 11, Avg(10): 19.30, Epsilon: 0.87
Episode: 30/100, Score: 21, Avg(10): 26.30, Epsilon: 0.77
Episode: 40/100, Score: 18, Avg(10): 15.20, Epsilon: 0.73
Episode: 50/100, Score: 17, Avg(10): 14.90, Epsilon: 0.69
Episode: 60/100, Score: 11, Avg(10): 17.30, Epsilon: 0.64
Episode: 70/100, Score: 11, Avg(10): 14.60, Epsilon: 0.61
Episode: 80/100, Score: 30, Avg(10): 16.00, Epsilon: 0.57
Episode: 90/100, Score: 14, Avg(10): 14.10, Epsilon: 0.55
Episode: 100/100, Score: 15, Avg(10): 14.60, Epsilon: 0.52

--- EXPERIMENTAL ARCHITECTURE RESULTS (3x64) ---
Total Episodes Run: 100
Overall Average Score: 17.36


**Results and Discussion: Experimental Architecture (3x64)**

**Experiment Logs**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-2b7s{text-align:right;vertical-align:bottom}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Episode</th>
    <th class="tg-7zrl">Score</th>
    <th class="tg-7zrl">Avg(10) Score</th>
    <th class="tg-7zrl">ϵ Value</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-2b7s">10</td>
    <td class="tg-2b7s">49</td>
    <td class="tg-2b7s">21.3</td>
    <td class="tg-2b7s">0.93</td>
  </tr>
  <tr>
    <td class="tg-2b7s">50</td>
    <td class="tg-2b7s">17</td>
    <td class="tg-2b7s">14.9</td>
    <td class="tg-2b7s">0.69</td>
  </tr>
  <tr>
    <td class="tg-2b7s">100</td>
    <td class="tg-2b7s">15</td>
    <td class="tg-2b7s">14.6</td>
    <td class="tg-2b7s">0.52</td>
  </tr>
</tbody>
</table>

**Final Result (100 Episodes):**

- **Total Episodes Run:** 100
- **Overall Average Score:** 17.36

**Analysis**

The Hypothesis was not supported by the 100-episode trial. The high-capacity experimental architecture (192 total neurons) failed to show clear signs of convergence, achieving a low overall average score of 17.36.

- **Instability and Slow Learning:** The per-episode scores and the 10-episode moving average (Avg(10)) remained low and highly volatile throughout the run. This lack of stable improvement suggests the agent's policy did not successfully begin exploiting useful strategies.
- **Implication of High Capacity:** The highly flexible, 3×64 network may be experiencing high variance. With a fixed, relatively small memory (2000 transitions) and a small batch_size (32), the increased capacity might be causing the network to overfit quickly to small batches of training data. This instability leads to poor generalization and a failure to build a robust policy.
- **Exploration:** By episode 100, the exploration rate ϵ was still at 0.52. This high level of random exploration combined with an unstable network prevented the agent from establishing a strong exploitation phase necessary to solve the environment.

**Conclusion:** 

For the simple CartPole environment, increasing network capacity without adjusting other critical hyperparameters—such as a larger Experience Replay Buffer or Training Frequency—can lead to poor performance and instability due to increased variance. The optimal architecture for this problem is likely closer to the simpler baseline.

## Follow-up Experiment: Adaptive Exploration Rate (ϵ-Greedy Scheduling)

**Theoretical Background: The Exploration-Exploitation Trade-off**

A core challenge in reinforcement learning is the **Exploration-Exploitation Trade-off**.

* **Exploration:** The agent tries new actions to gain a better understanding of the environment and potentially discover optimal strategies.
* **Exploitation:** The agent chooses the action currently believed to yield the highest reward, based on its current knowledge.

The $\epsilon$-greedy strategy addresses this by setting a probability $\epsilon$ for random action (exploration) and a probability $1-\epsilon$ for selecting the optimal known action (exploitation).

**Decay Schedule and Non-Stationarity**

Traditionally, $\epsilon$ starts high (full exploration) and decays exponentially to a minimum value ($\epsilon_{\text{min}}$). However, a fixed decay rate may be inefficient:

* If the agent learns quickly, a slow decay rate wastes time on unnecessary exploration.
* If the environment is non-stationary (e.g., changes over time, though not in CartPole), the agent might stop exploring too soon.

**Adaptive $\epsilon$ decay** attempts to make the decay rate sensitive to the agent's performance, allowing it to rapidly shift to exploitation once a high-performing policy is found. This balances the need for initial exploration with the urgency of utilizing learned knowledge efficiently.

**Experiment Objective and Instructions**

- **Objective:** To implement and test an **adaptive $\epsilon$ decay strategy** to dynamically balance exploration and exploitation, and observe its effect on convergence speed.
- **Adaptive Strategy:** Modify the decay to accelerate reduction of $\epsilon$ (using a larger multiplier like $0.9$ instead of $0.995$) if the agent achieves a high score (e.g., $\ge 200$) in an episode, signifying successful learning.

This content is now ready to be pasted directly into a markdown cell in your Jupyter Notebook.

In [ ]:
# Function to adjust epsilon based on performance
def adjust_epsilon(score, consecutive_success_threshold=200):
    global epsilon 
    global epsilon_min
    global epsilon_decay

    if score >= consecutive_success_threshold: 
        # Reduce epsilon faster (0.9 multiplier) if performance is good
        epsilon = max(epsilon_min, epsilon * 0.9)  
    else: 
        # Regular epsilon decay
        epsilon = max(epsilon_min, epsilon * epsilon_decay)  

# IMPORTANT: Reset epsilon and environment before starting the new experiment
epsilon = 1.0 
episodes = 20 # Run for 20 episodes to observe effect (or adjust as needed)

# Train the model with adaptive epsilon decay
print(f"Starting Adaptive Epsilon Training for {episodes} episodes...")
for e in range(episodes): 
    # Robust Reset Handler
    reset_result = env.reset()
    state = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    # Reshape using the globally defined state_size
    state = np.reshape(state, [1, state_size])  

    total_reward = 0 
    time = 0

    while time < 500:  # Limit the episode to 500 time steps
        action = act(state)  
        
        # Robust Step Handler
        step_result = env.step(action)
        
        if len(step_result) == 5:
            next_state, reward, terminated, truncated, _ = step_result
            done = terminated or truncated
        else: 
            next_state, reward, done, _ = step_result
            truncated = False 

        # Penalize for reaching a terminal state
        reward = reward if not (done or truncated) else -10  
        total_reward += reward 

        # Reshape next_state
        next_state = np.reshape(next_state, [1, state_size]) 

        remember(state, action, reward, next_state, done or truncated) 
        state = next_state 
        time += 1

        if done or truncated:
            adjust_epsilon(time) # Adjust based on the score (time survived)
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.4f}")  
            break

        if len(memory) > batch_size and time % 10 == 0:  
            replay(batch_size)  

Starting Adaptive Epsilon Training for 20 episodes...
Episode: 1/20, Score: 42, Epsilon: 0.9752
Episode: 2/20, Score: 18, Epsilon: 0.9655
Episode: 3/20, Score: 12, Epsilon: 0.9559
Episode: 4/20, Score: 16, Epsilon: 0.9464
Episode: 5/20, Score: 19, Epsilon: 0.9369
Episode: 6/20, Score: 23, Epsilon: 0.9229
Episode: 7/20, Score: 20, Epsilon: 0.9137
Episode: 8/20, Score: 16, Epsilon: 0.9046
Episode: 9/20, Score: 19, Epsilon: 0.8956
Episode: 10/20, Score: 16, Epsilon: 0.8867
Episode: 11/20, Score: 21, Epsilon: 0.8734
Episode: 12/20, Score: 11, Epsilon: 0.8647
Episode: 13/20, Score: 17, Epsilon: 0.8561
Episode: 14/20, Score: 37, Epsilon: 0.8391
Episode: 15/20, Score: 23, Epsilon: 0.8266
Episode: 16/20, Score: 24, Epsilon: 0.8142
Episode: 17/20, Score: 11, Epsilon: 0.8061
Episode: 18/20, Score: 11, Epsilon: 0.7981
Episode: 19/20, Score: 30, Epsilon: 0.7862
Episode: 20/20, Score: 11, Epsilon: 0.7783


**Results and Discussion: Adaptive Exploration Rate (ϵ-Greedy Scheduling)**

The follow-up experiment tested an Adaptive ϵ Decay strategy, designed to accelerate the shift from exploration to exploitation upon reaching a high-performance threshold (score ≥200). Since the agent was run for only 20 episodes, the primary observation is the pattern of decay.

**Experiment Logs**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-2b7s{text-align:right;vertical-align:bottom}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Episode</th>
    <th class="tg-7zrl">Score (Time Steps)</th>
    <th class="tg-7zrl">ϵ Value</th>
    <th class="tg-7zrl">Decay Type</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-2b7s">1</td>
    <td class="tg-2b7s">42</td>
    <td class="tg-2b7s">0.9752</td>
    <td class="tg-7zrl">Regular</td>
  </tr>
  <tr>
    <td class="tg-2b7s">6</td>
    <td class="tg-2b7s">23</td>
    <td class="tg-2b7s">0.9229</td>
    <td class="tg-7zrl">Regular</td>
  </tr>
  <tr>
    <td class="tg-2b7s">14</td>
    <td class="tg-2b7s">37</td>
    <td class="tg-2b7s">0.8391</td>
    <td class="tg-7zrl">Regular</td>
  </tr>
  <tr>
    <td class="tg-2b7s">20</td>
    <td class="tg-2b7s">11</td>
    <td class="tg-2b7s">0.7783</td>
    <td class="tg-7zrl">Regular</td>
  </tr>
</tbody>
</table>

**Analysis**

The agent ran for 20 episodes and achieved a final ϵ value of 0.7783.

- **Absence of Adaptive Trigger:** Crucially, the maximum score achieved in any single episode was 42 steps (Episode 1). Since this score is significantly below the set success threshold of 200 steps, the rapid, performance-based decay (ϵ×0.9) was never triggered.
- **Regular Decay Dominance:** The agent relied entirely on the standard exponential decay (ϵ×0.995 for each episode). This is reflected in the steady, slow reduction of ϵ from 1.0 down to 0.7783.
- **Implications for Adaptive Scheduling:** This result highlights a key limitation in implementing simple adaptive schedules: the adaptive mechanism is only useful after the agent has successfully learned an initial, stable policy. Given the instability observed with the 3×64 network architecture in the previous experiment, the agent requires more training episodes to reach the 200-step threshold before the adaptive scheduler can influence the learning rate.

**Conclusion:** 

The adaptive decay mechanism's effect on convergence speed could not be assessed in this short run, as the agent failed to reach the performance threshold required to activate the accelerated decay. The primary decay observed was the regular exponential decay, indicating that the agent remained heavily in the exploration phase throughout the 20 episodes.

## Follow-up Experiment: Custom Reward Function (Reward Shaping)

**Theoretical Background: Reward Sparsity and Shaping**

The default CartPole reward is sparse—the agent receives $+1$ for every time step the pole is balanced, and the episode ends upon failure. This signal is minimal.

**Reward Shaping** is the technique of designing a custom reward function $R'(s,a)$ that provides the agent with more informative feedback before the terminal state is reached. This is done by adding a potential-based auxiliary reward $F(s,a)$ to the original reward $R(s,a)$:

$$R'(s,a) = R(s,a) + F(s,a)$$

**The Importance of Potential-Based Shaping**

To guarantee that the optimal policy of the shaped environment remains the same as the original environment, the shaping function $F(s,a)$ should be **potential-based**, meaning it is derived from a scalar potential function $\Phi(s)$:

$$F(s,a) = \gamma\Phi(s') - \Phi(s)$$

Where $\gamma$ is the discount factor, $s$ is the current state, and $s'$ is the next state.

In this experiment, we implement a simple, **non-potential-based shaping function** that leverages the normalized state variables (pole angle $\theta$ and cart position $x$) to immediately reward stability and centering. While not strictly potential-based, this form of guidance is often used heuristically to accelerate learning in preliminary stages.

**Experiment Objective and Instructions**

- **Objective:** To understand the impact of providing dense, continuous feedback through reward shaping on the agent's learning speed and stability, compared to the default sparse reward.
- **Custom Function:** Implement a reward function that rewards the agent proportionally to $1 - |x|$ (cart distance from center) and $1 - |\theta|$ (pole angle from vertical), with a higher weight given to maintaining a small pole angle.

In [ ]:
# Define a custom reward function based on the cart position and pole angle
def custom_reward(state):
    # Extract state variables: x (cart position), x_dot (cart velocity), 
    # theta (pole angle), theta_dot (pole angular velocity)
    # Note: State is passed as a 1D numpy array when called from the main loop
    x, x_dot, theta, theta_dot = state.flatten()
    
    # CartPole environment limits:
    # Max Cart Position: 2.4 (failure at $|x| \ge 2.4$)
    # Max Pole Angle: 0.20948 radians (approx 12 degrees) (failure at $| \theta | \ge 0.20948$)
    
    # Normalize position and angle to be between 0 and 1, where 1 is optimal
    # Closer to the optimal value (0) results in a value closer to 1
    position_reward = 1.0 - (abs(x) / 2.4) 
    angle_reward = 1.0 - (abs(theta) / 0.20948) 
    
    # Combined reward: Prioritize keeping the pole upright (angle)
    # Weights: 60% for angle, 40% for position
    reward = 0.6 * angle_reward + 0.4 * position_reward
    
    # Minimum small positive reward to encourage survival, even when suboptimal
    return max(0.01, reward) 

# IMPORTANT: Reset epsilon and environment before starting the new experiment
epsilon = 1.0 
episodes = 20 # Run for 20 episodes to observe effect

# Train the model with the custom reward function
print(f"Starting Custom Reward Training for {episodes} episodes...")
for e in range(episodes): 
    # Robust Reset Handler
    reset_result = env.reset()
    state = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    state = np.reshape(state, [1, state_size])  
    
    time = 0
    done = False
    
    while time < 500:  # Limit the episode to 500 time steps
        action = act(state)  
        
        # Robust Step Handler
        step_result = env.step(action)
        
        if len(step_result) == 5:
            next_state_raw, reward_default, terminated, truncated, _ = step_result
            done = terminated or truncated
        else: # Assumes len is 4
            next_state_raw, reward_default, done, _ = step_result
            truncated = False

        # REWARD SHAPING: Calculate custom reward based on the raw state
        # Only use custom reward if the episode is still running
        reward = custom_reward(next_state_raw) if not (done or truncated) else -10
        
        # Reshape the raw next_state for the Q-Network
        next_state = np.reshape(next_state_raw, [1, state_size]) 

        # Store the experience in memory
        remember(state, action, reward, next_state, done or truncated)
        state = next_state  
        time += 1

        if done or truncated:
            # Use regular, non-adaptive epsilon decay for simplicity in this experiment
            global epsilon 
            epsilon = max(epsilon_min, epsilon * epsilon_decay)
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.4f}")
            break

        if len(memory) > batch_size and time % 10 == 0:  
            replay(batch_size)

Starting Custom Reward Training for 20 episodes...
Episode: 1/20, Score: 14, Epsilon: 0.9900
Episode: 2/20, Score: 32, Epsilon: 0.9704
Episode: 3/20, Score: 31, Epsilon: 0.9511
Episode: 4/20, Score: 32, Epsilon: 0.9322
Episode: 5/20, Score: 13, Epsilon: 0.9229
Episode: 6/20, Score: 26, Epsilon: 0.9092
Episode: 7/20, Score: 34, Epsilon: 0.8911
Episode: 8/20, Score: 21, Epsilon: 0.8778
Episode: 9/20, Score: 15, Epsilon: 0.8691
Episode: 10/20, Score: 24, Epsilon: 0.8561
Episode: 11/20, Score: 16, Epsilon: 0.8475
Episode: 12/20, Score: 18, Epsilon: 0.8391
Episode: 13/20, Score: 16, Epsilon: 0.8307
Episode: 14/20, Score: 13, Epsilon: 0.8224
Episode: 15/20, Score: 15, Epsilon: 0.8142
Episode: 16/20, Score: 29, Epsilon: 0.8021
Episode: 17/20, Score: 21, Epsilon: 0.7901
Episode: 18/20, Score: 10, Epsilon: 0.7862
Episode: 19/20, Score: 15, Epsilon: 0.7783
Episode: 20/20, Score: 12, Epsilon: 0.7705


**Results and Discussion: Custom Reward Function (Reward Shaping)**

This experiment introduced a custom, dense reward function that provided granular feedback to the agent on its proximity to the optimal state (pole vertical, cart centered). This method of Reward Shaping aimed to accelerate learning compared to the default sparse +1 reward.

**Experiment Logs**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-2b7s{text-align:right;vertical-align:bottom}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Episode</th>
    <th class="tg-7zrl">Score (Time Steps)</th>
    <th class="tg-7zrl">ϵ Value</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-2b7s">1</td>
    <td class="tg-2b7s">14</td>
    <td class="tg-2b7s">0.99</td>
  </tr>
  <tr>
    <td class="tg-2b7s">7</td>
    <td class="tg-2b7s">34</td>
    <td class="tg-2b7s">0.8911</td>
  </tr>
  <tr>
    <td class="tg-2b7s">10</td>
    <td class="tg-2b7s">24</td>
    <td class="tg-2b7s">0.8561</td>
  </tr>
  <tr>
    <td class="tg-2b7s">17</td>
    <td class="tg-2b7s">21</td>
    <td class="tg-2b7s">0.7901</td>
  </tr>
  <tr>
    <td class="tg-2b7s">20</td>
    <td class="tg-2b7s">12</td>
    <td class="tg-2b7s">0.7705</td>
  </tr>
</tbody>
</table>

**Analysis**

The agent ran for 20 episodes using the custom reward function and the high-capacity 3×64 Q-Network.

- **Volatile Performance:** The performance remained highly volatile, similar to the first architecture experiment. Scores jumped significantly between episodes (e.g., Episode 7 reached 34 steps, but Episode 18 immediately dropped to 10 steps). The agent failed to establish a consistent, improving policy within the 20 episodes.
- **Reward Shaping Impact:** Theoretically, the custom reward—which weighted the pole angle (60%) more heavily than the cart position (40%)—should provide a much clearer gradient for the Q-Network to follow, immediately rewarding actions that stabilize the pole.
- **Dominance of Capacity Issue:** The lack of clear, sustained improvement suggests that the instability of the high-capacity 3×64 Q-Network remains the dominant factor. Reward shaping provides a better training signal, but it cannot fully compensate for a model that suffers from high variance and is likely overfitting to small mini-batches from the replay buffer. The model is too powerful and flexible for the limited and noisy early-training data.

**Conclusion:** 

While reward shaping is a valuable technique for solving problems with sparse reward signals, this experiment demonstrates that its benefits can be obscured by an unstable underlying network architecture. The model capacity problem (high variance) appears to be a more fundamental hurdle in this setup than the reward sparsity problem, requiring architectural changes or adjustments to the training regime (e.g., increasing the replay buffer size) before the benefits of dense reward shaping can be fully realized.

# Conclusion

The Deep Q-Learning (DQN) project successfully implemented the core components of a DQN agent for the CartPole-v1 environment, including Q-network approximation, experience replay, and the ϵ-greedy strategy. The project served as a foundational exploration into how key design parameters—network architecture, exploration scheduling, and reward functions—impact the learning process.

**Key Findings and Performance Assessment**

Despite successfully building the learning pipeline, the agent demonstrated suboptimal performance across all tested configurations, failing to consistently solve the CartPole-v1 task (scores were typically below 50, far from the target of ∼200).

The specific experimental findings highlighted critical instability issues:

- **Network Capacity:** The higher-capacity 3×64 experimental architecture did not outperform the 2×32 baseline, suggesting the larger network introduced high variance and potential overfitting given the limited training data and episodes (100).
- **Exploration:** The adaptive ϵ decay strategy was ineffective because the agent's performance rarely met the high score threshold (200) required to trigger the faster decay, resulting in sustained, slow exploration.
- **Reward Shaping:** Providing a dense, custom reward signal, while slightly increasing some peak scores, was not enough to overcome the instability of the high-capacity Q-network, indicating that the network architecture problem was the dominant limiting factor.

**Challenges and Recommendations**

The project identified three primary areas for future work necessary to achieve robust convergence:

- **Convergence & Training:** The low scores and high volatility strongly suggest the need to increase training episodes (recommended 500–1000) and potentially adjust the learning rate to allow the Q-function to converge reliably.
- **Stability:** The agent's instability necessitates the integration of advanced DQN techniques, specifically Double DQN to counteract Q-value overestimation, and Prioritized Experience Replay to focus training on the most informative transitions.
- **Reward Function:** Future reward shaping experiments should prioritize potential-based methods to ensure the optimal policy of the shaped environment remains consistent with the original task.

In summary, the project provides a strong framework for DQN implementation and valuable empirical insight into the challenges of parameter tuning. Achieving mastery of the CartPole environment requires further optimization of hyperparameters and the adoption of more stable and sophisticated DQN variants.